# Fine tuning an Italian GPT on the 'Divine Comedy'

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch 
import urllib.request
import re 
import torch.nn.functional as F
import math 


url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
with urllib.request.urlopen(url) as response:
    dante_text = response.read().decode('utf-8')

model_name = "LorenzoDeMattei/GePpeTto"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(30000, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=30000, bias=False)
)


### 1- Data cleaning

In [32]:
# Canto headers, e.g. "Inferno: Canto I", "Purgatorio: Canto XXXIII"
canto_header_re = re.compile(r'^(Inferno|Purgatorio|Paradiso):\s*Canto\s+[IVXLCDM]+\s*$')

# Front-matter title lines
title_lines = {
    "LA DIVINA COMMEDIA",
    "di Dante Alighieri",
    "INFERNO",
    "PURGATORIO",
    "PARADISO",
}

def clean_editorial_lines(text):
    cleaned = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped in title_lines:
            continue
        if canto_header_re.match(stripped):
            continue
        cleaned.append(line)
    return '\n'.join(cleaned)

dante_text = clean_editorial_lines(dante_text)
print(len(dante_text)) 

534889


In [33]:
# sanity check
text = "Nel mezzo del cammin di nostra vita"
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)


print(tokenizer)
print(tokens)
print(ids)

GPT2Tokenizer(name_or_path='LorenzoDeMattei/GePpeTto', vocab_size=30000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
['Nel', 'Ġmezzo', 'Ġdel', 'Ġcamm', 'in', 'Ġdi', 'Ġnostra', 'Ġvita']
[1771, 2312, 280, 4881, 266, 272, 1930, 991]


### 1 - One forward pass, by hand

Manually walking logits -> softmax -> argmax -> top-k, before relying on any HF convenience methods.

In [34]:
prompt = "Nel mezzo del cammin di nostra vita"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print(inputs["input_ids"].shape)
print(outputs.logits.shape)  # (batch, sequence, vocab)


torch.Size([1, 8])
torch.Size([1, 8, 30000])


In [35]:
# Prediction for the next token
next_token_logits = outputs.logits[:, -1, :]
probs = F.softmax(next_token_logits, dim=-1)
print(probs.sum())  # sanity check: should be 1.0

next_token_id = torch.argmax(probs, dim=-1)
print(tokenizer.decode(next_token_id))

top_probs, top_ids = torch.topk(probs, 10)
for prob, token_id in zip(top_probs[0], top_ids[0]):
    print(f"{tokenizer.decode(token_id)!r}: {prob.item():.4f}")

tensor(1.0000)
,
',': 0.3417
' c': 0.0516
'.': 0.0515
' ci': 0.0433
' e': 0.0323
' non': 0.0254
' si': 0.0240
' è': 0.0152
' il': 0.0129
' vi': 0.0122


### 2 - Greedy decoding demo

prompt -> next token -> append -> predict again -> append -> ...
This loop is the basic mechanism behind text generation, before using `model.generate()`.

In [36]:
generated = inputs["input_ids"]

for _ in range(50):
    with torch.no_grad():
        outputs = model(generated)
    next_token_logits = outputs.logits[:, -1, :]
    next_token_id = torch.argmax(next_token_logits, dim=-1)
    generated = torch.cat([generated, next_token_id.unsqueeze(0)], dim=1)

print(tokenizer.decode(generated[0]))

Nel mezzo del cammin di nostra vita, il nostro cuore è stato inondato di luce. Il nostro cuore è stato inondato di luce. Il nostro cuore è stato inondato di luce. Il nostro cuore è stato inondato di luce. Il nostro cuore è stato inondato di luce


---
# Fine tuning the model


In [37]:
dante_ids = tokenizer.encode(dante_text)

# Data spltting same as trasformer

data = torch.tensor(dante_ids, dtype=torch.long)

n1 = int(0.9 * len(data))
n2 = int(0.95 * len(data))

train = data[:n1]
val = data[n1:n2]
test = data[n2:]

### 4 - Batching

In [38]:
def get_batch(split, batch_size, block_size):
    if split == 'train':
        data_split = train
    elif split == 'val':
        data_split = val
    elif split == 'test':
        data_split = test
    else:
        raise ValueError('Split must be "train", "val" or "test"')

    ix = torch.randint(0, len(data_split) - block_size, (batch_size,))

    X = torch.stack([
        data_split[index : index + block_size] for index in ix
    ])  # input tokens

    Y = torch.stack([
        data_split[index + 1 : index + block_size + 1] for index in ix
    ])  # same sequence shifted one token to the right

    return X, Y

### 6 - Hyperparameters

In [39]:
batch_size = 4
block_size = 128
learning_rate = 5e-5 # Model already pretrained
max_steps = 100
eval_interval = 10


xb, yb = get_batch('train', batch_size, block_size)
print(xb.shape, yb.shape)
print(tokenizer.decode(xb[0]))

torch.Size([4, 128]) torch.Size([4, 128])
 come si scalappia,
perché ci trema e di che congaudete.
  Ora chi fosti, piacciati ch'io sappia,
e perché tanti secoli giaciuto
qui se', ne le parole tue mi cappia".
  "Nel tempo che 'l buon Tito, con l'aiuto
del sommo rege, vendicò le fóra
ond'uscì 'l sangue per Giuda venduto,
  col nome che più dura e più onora
era io di là", rispuose quello spirto,
"famoso assai, ma non con fede ancora.



### 7 - Device placement
This is done so that the notebook can run on Colab.

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = model.to(device)

cpu


### 10 - Baseline loss check

Loss on the *untouched* pretrained model, before any gradient steps.



In [41]:
xb, yb = get_batch('train', batch_size, block_size)
xb, yb = xb.to(device), yb.to(device)

with torch.no_grad():
    outputs = model(input_ids=xb, labels=yb)

loss = outputs.loss.item()
print(f"Baseline loss: {loss:.4f}")
print(f"Baseline perplexity: {math.exp(loss):.2f}")

Baseline loss: 9.8947
Baseline perplexity: 19825.06


A loss around 9-10 here is expected: GePpeTto was pretrained on modern Italian
Wikipedia, and the Commedia is 700 years old, with archaic syntax and poetic structure.
Fine-tuning is meant to close that gap.

### 8 - Evaluation function

In [42]:
@torch.no_grad()
def estimate_loss(splits, eval_iters=10):
    model.eval()
    losses = {}
    for split in splits:
        split_losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split, batch_size, block_size)
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(input_ids=xb, labels=yb)
            split_losses.append(outputs.loss.item())
        losses[split] = sum(split_losses) / len(split_losses)
    model.train()
    return losses

### 13 - Optimizer + training loop

Optimizer is created fresh here, right before the loop starts — no demo cell has touched
the model or optimizer state beforehand (see section 10).

In [43]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_steps):
    model.train()

    xb, yb = get_batch('train', batch_size, block_size)
    xb, yb = xb.to(device), yb.to(device)

    outputs = model(input_ids=xb, labels=yb)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        losses = estimate_loss(['train', 'val'], eval_iters=10)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

step 0: train loss 8.0897, val loss 8.0707
step 10: train loss 6.6131, val loss 6.6163
step 20: train loss 6.3614, val loss 6.4152
step 30: train loss 6.2445, val loss 6.3057
step 40: train loss 6.0705, val loss 6.2229
step 50: train loss 6.0365, val loss 6.0605
step 60: train loss 5.9264, val loss 6.0874
step 70: train loss 5.8815, val loss 5.9332
step 80: train loss 5.7819, val loss 5.9627
step 90: train loss 5.8050, val loss 5.8390


### 14 - Saving model + tokenizer

In [44]:
save_path = "./gepPetto-dante"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gepPetto-dante/tokenizer_config.json', './gepPetto-dante/tokenizer.json')

### 15 - Qualitative sampling

In [45]:
def generate_text(prompt, max_new_tokens=150, temperature=0.8, top_k=50):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

prompts = [
    "Nel mezzo del cammin di nostra vita",
    "Nel mezzo",
    "Amor che ne la mente mi ragiona",
]

for p in prompts:
    print(generate_text(p))
    print("---")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nel mezzo del cammin di nostra vita
 di, chel nonò'r, mète ',
'' di' suor e' a' suoi'
 chel suo non laltrol
 e lel si in; e'' di tua che?".  "sì sì?  'l' che dil si,di,chever sice"    O la in " è,''
'è la''' suo
' la l,, ilr,' e'
lè tua che, mir, che, per
 che mirn in'". ò la donna ",' limp, che' si,' me' non
', di che' a che'
---


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Nel mezzo del vigna dar lultimo duno
' de'r che shave diòl
' in sua salto che la
'lalta cha terra' a ilarco
 suor non è la, per a, fa'
''' a di.
 O a' non mi lio' chel
'' di' mè, comio miiote'
' di non chio non,' ' e' un.  ' maestro mio' tu chel mi a

'''vi mio chevi in, tua,'', non ti
 chio e
' per che tu di, timi' non mi' per in,
 non mi
---
Amor che ne la mente mi ragiona
 lo ' fa, se'', di". ò'dio:Or di,':Tu' titi
 tu' mei:Si'' in, meti, sì'',ma'' me,e
 puoi non puoi tu non che si''ga? Lei:La, tu mirai
''' mi diè se' me che si. U:Vio' sono'' là,ma non' mai
 a ti di; tu me chi non'''' che tu fossi di,che'' mi'
 nonrhai; non'vi né che non'''
' puoi.  a che tu milhai
---


In [46]:
### 16 - Final test metrics

final_losses = estimate_loss(['train', 'val', 'test'], eval_iters=20)
test_perplexity = math.exp(final_losses['test'])

print("Test loss:", final_losses['test'])
print("Test perplexity:", test_perplexity)

Test loss: 5.907018089294434
Test perplexity: 367.6083441783239


# Tracking

## Experiment 0 - baseline (no LoRA)

- BPE tokenization (GePpeTto GPT-2 tokenizer, vocab = 30000)
- block_size = 128
- batch_size = 4
- learning_rate = 5e-5
- max_steps = 100

Final test loss ~ 5.90
Final test perplexity ~ 367.60

**Outcome:** Baseline, no rhymes and gibberish Italian: 
Nel mezzo del cammin di nostra vita
 di, chel nonò'r, mète ',
'' di' suor e' a' suoi'